# TopAGNPS import verification — Ohio dataset

Replays the **full import pipeline on real TopAGNPS output files** (the Ohio
dataset loads in seconds, unlike the Goodwin 1 m raster which takes ~18 min).
For each coarsening level it:

1. imports the CSVs and reads the reach-ID raster (geometry + US/DS orientation),
2. assigns Strahler order and optionally drops low-order reaches,
3. resamples, assigns cross sections, adjusts the thalweg,
4. builds the CCHE1D network and validates it (errors must be empty),
5. **re-derives the tw_cell and inflow_reaches records from the generic
   reach/node state and compares them to the DataFrames** (this catches wrong
   node references without hard-coding dataset ids),
6. writes the files, checks the `tw_cell` file format, and round-trips.

The last cell shows how to run the same verification on Goodwin Creek.

`STRَاهLER_DROP = 1` in the pipeline cell switches between "full network"
(0) and "coarsened" (1, 2).

## 0. Setup

In [ ]:
import sys
import copy
import time
from pathlib import Path
from collections import Counter

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import topagnps2cche1d as t2c
from topagnps2cche1d.network import read_cche1d_network, validate_network
import topagnps2cche1d.network as netmod

DATA = ROOT / "data_inputs" / "topagnps_ohio_files"
assert (DATA / "AnnAGNPS_Reach_IDs.asc").exists(), "Ohio dataset not found"
print("dataset:", DATA)

In [ ]:
t0 = time.time()
w0 = t2c.Watershed()
w0.import_topagnps_reaches_network(DATA / "AnnAGNPS_Reach_Data_Section.csv")
w0.import_topagnps_cells(DATA / "AnnAGNPS_Cell_Data_Section.csv")
w0.read_reaches_geometry_from_topagnps_asc_file(DATA / "AnnAGNPS_Reach_IDs.asc")
w0.assign_strahler_number_to_reaches()
print(f"import done in {time.time()-t0:.1f}s: {len(w0.reaches)} reaches, "
      f"{len(w0.cells)} cells, max strahler {w0.get_highest_strahler_number()}")

In [ ]:
def pipeline(drop, step=10):
    """Fresh clone of the import + one full pipeline run at a coarsening level."""
    w = t2c.Watershed()
    w.reaches = {rid: copy.deepcopy(r) for rid, r in w0.reaches.items()}
    w.cells = copy.deepcopy(w0.cells)
    w.geomatrix = copy.deepcopy(w0.geomatrix)
    w.update_graph()
    w.assign_strahler_number_to_reaches()
    if drop:
        w.ignore_reaches_with_strahler_leq(drop)
    w.update_watershed()
    w.resample_reaches(id_list="all", step=step)
    w.assign_cross_section_to_all_points()
    w.adjust_cross_sections_elevation_with_slope()
    net = w.build_cche1d_network(strahler_drop=0)
    return w, net

STR_DROP = 1   # 0 = full network, 1 or 2 = coarsened
w, net = pipeline(STR_DROP)
errors, warnings = validate_network(net)
print(f"drop<={STR_DROP}: kept={len(w.current_graph)} "
      f"nodes={len(net.nodes)} types={dict(sorted(Counter(n.nd_type for n in net.nodes).items()))}")
assert errors == [] and warnings == []
print("validate_network: PASS")

In [ ]:
# Bookkeeping: re-derive the records from the generic nodes and compare with
# the DataFrames.  Any mismatch means a node reference is wrong.
valid_nd = {n.nd_id for n in net.nodes}
origin = {gid: nid for nid, gid in net.node_origin.items() if gid is not None}

expected_in = []
for rid in w.current_graph:
    for node in w.reaches[rid].nodes.values():
        for s in node.inflow_reach_sources:
            expected_in.append((s, origin.get(node.id, node.id)))
df_in = w.get_list_of_inflow_reaches_df()
got_in = ([(int(a), int(b)) for a, b in
           df_in[["inflow_topagnps_reach_id", "receiving_node_id"]].itertuples(index=False)]
          if not df_in.empty else [])
print(f"removed-reach records: expected={len(expected_in)} df={len(got_in)}")
print("match:", sorted(expected_in) == sorted(got_in))
assert sorted(expected_in) == sorted(got_in)
assert set(b for _, b in got_in) <= valid_nd

n_cell_rows = sum(len(node.inflow_cell_sources)
                  for rid in w.current_graph
                  for node in w.reaches[rid].nodes.values())
df_tw = w.create_cche1d_twcells_df()
print(f"tw_cell rows: expected={n_cell_rows} df={len(df_tw)}")
assert len(df_tw) == n_cell_rows
assert set(df_tw["TW_NDDSID"]) <= valid_nd
print("total drainage area (m^2):", df_tw["TW_DRAREA"].sum())

In [ ]:
# Write, check the file format, round-trip.
out = Path(f"output_ohio_drop{STR_DROP}")
out.mkdir(exist_ok=True)
paths = w.write_cche1d_dat_files(casename=f"ohio_d{STR_DROP}", output_folder=str(out))
for k, p in sorted(paths.items()):
    print(f"  {p.name:40s} {p.stat().st_size:>8d} bytes")

lines = (out / f"ohio_d{STR_DROP}_tw_cell.dat").read_text().splitlines()
print("tw_cell file: header count", lines[0], "rows", len(lines) - 2)
assert int(lines[0]) == len(df_tw)
assert lines[1].startswith("TW_ID\tTW_NUMB\tTW_NDDSID")

back = read_cche1d_network(out)
errors2, warnings2 = validate_network(back)
print("round-trip:", len(back.nodes), "nodes ->",
      "PASS" if not errors2 and not warnings2 else (errors2, warnings2))
assert not errors2 and not warnings2

## Goodwin Creek (same verification, slow)

The Goodwin 1 m raster takes ~18 min to load.  Run the identical pipeline with
the Goodwin paths (in a fresh kernel or after the cells above are done):

```python
DATA = ROOT / "data_inputs" / "goodwin_creek_1m"
w0 = t2c.Watershed()
w0.import_topagnps_reaches_network(DATA / "AnnAGNPS_Reach_Data_Section.csv")
w0.import_topagnps_cells(DATA / "AnnAGNPS_Cell_Data_Section.csv")
w0.read_reaches_geometry_from_topagnps_asc_file(DATA / "AnnAGNPS_Reach_IDs.asc")
w0.assign_strahler_number_to_reaches()
# then: w, net = pipeline(max_strahler - 2)  # the usual coarsening
# and the same assertions as above.
```
